## Welcome to Week 4, Day 4

This is the start of an AWESOME project! Really simple and very effective.

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_core.tools import Tool

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver


In [2]:
load_dotenv(override=True)

True

### Asynchronous LangGraph

To run a tool:  
Sync: `tool.run(inputs)`  
Async: `await tool.arun(inputs)`

To invoke the graph:  
Sync: `graph.invoke(state)`  
Async: `await graph.ainvoke(state)`

In [3]:
class State(TypedDict):
    
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

In [4]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

## Extra installation step - if you don't have Node and Playwright on your computer

Next, you need to install NodeJS and Playwright on your computer if you don't already have them. Please see instructions here:

[Node and Playwright setup](../setup/SETUP-node.md)

## And now - after Installing Playwright, a heads up for Windows PC Users:

While executing the next few cells, you might hit a problem with the Playwright browser raising a NotImplementedError.

This should work when we move to python modules, but it can cause problems in Windows in a notebook.

If you it this error and would like to run the notebook, you need to make a small change which seems quite hacky! You need to do this AFTER installing Playwright (prior cells)

1. Right click in `.venv` in the File Explorer on the left and select "Find in folder"
2. Search for `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())`  
3. That code should be found in a line of code in a file called `kernelapp.py`
4. Comment out the entire else clause that this line is a part of - see the fragment below. Be sure to have the "pass" statement after the ImportError line.
5. Restart the kernel by pressing the "Restart" button above

```python
        if sys.platform.startswith("win") and sys.version_info >= (3, 8):
            import asyncio
 
            try:
                from asyncio import WindowsProactorEventLoopPolicy, WindowsSelectorEventLoopPolicy
            except ImportError:
                pass
                # not affected
           # else:
            #    if type(asyncio.get_event_loop_policy()) is WindowsProactorEventLoopPolicy:
                    # WindowsProactorEventLoopPolicy is not compatible with tornado 6
                    # fallback to the pre-3.8 default of Selector
                    # asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
```

Thank you to student Nicolas for finding this, and to Kalyan, Yaki, Zibin and Bhaskar for confirming that this worked for them! And to Vladislav for the extra pointers.

As an alternative, you can just move to a Python module (which we do anyway in Day 5)

In [5]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

# Fix: uvicorn.server imports asyncio.run as a local name (asyncio_run) at import time.
# Newer uvicorn calls it with loop_factory= but nest_asyncio's patched version rejects that.
# We must patch uvicorn.server's own module-level reference directly.
import uvicorn.server as _uvicorn_server
_orig_asyncio_run = _uvicorn_server.asyncio_run
def _patched_asyncio_run(main, **kwargs):
    kwargs.pop('loop_factory', None)
    return _orig_asyncio_run(main, **kwargs)
_uvicorn_server.asyncio_run = _patched_asyncio_run


### The LangChain community

One of the remarkable things about LangChain is the rich community around it.

Check this out:


In [6]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser

# If you get a NotImplementedError here or later, see the Heads Up at the top of the notebook

async_browser =  create_async_playwright_browser(headless=False)  # headful mode
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [7]:
for tool in tools:
    print(f"{tool.name}={tool}")

click_element=async_browser=<Browser type=<BrowserType name=chromium executable_path=C:\Users\rishu.kumar\AppData\Local\ms-playwright\chromium-1187\chrome-win\chrome.exe> version=140.0.7339.16>
navigate_browser=async_browser=<Browser type=<BrowserType name=chromium executable_path=C:\Users\rishu.kumar\AppData\Local\ms-playwright\chromium-1187\chrome-win\chrome.exe> version=140.0.7339.16>
previous_webpage=async_browser=<Browser type=<BrowserType name=chromium executable_path=C:\Users\rishu.kumar\AppData\Local\ms-playwright\chromium-1187\chrome-win\chrome.exe> version=140.0.7339.16>
extract_text=async_browser=<Browser type=<BrowserType name=chromium executable_path=C:\Users\rishu.kumar\AppData\Local\ms-playwright\chromium-1187\chrome-win\chrome.exe> version=140.0.7339.16>
extract_hyperlinks=async_browser=<Browser type=<BrowserType name=chromium executable_path=C:\Users\rishu.kumar\AppData\Local\ms-playwright\chromium-1187\chrome-win\chrome.exe> version=140.0.7339.16>
get_elements=async_b

In [8]:
tool_dict = {tool.name:tool for tool in tools}

navigate_tool = tool_dict.get("navigate_browser")
extract_text_tool = tool_dict.get("extract_text")

    
await navigate_tool.arun({"url": "https://news.google.com/home?hl=en-IN&gl=IN&ceid=IN%3Aen"})
text = await extract_text_tool.arun({})


In [9]:
import textwrap
print(textwrap.fill(text))

Google News News Google News News Advanced search Narrow your search
results Exact phrase Has words Exclude words Website Enter a valid web
address Date Any time Any time Past hour Past 24 hours Past week Past
year Clear Search Help Help Privacy Terms About Google Get the Android
app Get the iOS app Send feedback Settings Settings Language & region
English (India) Sign in Home For you Following News Showcase India
World Local Business Technology Entertainment Sports Science Health
More Your briefing Thursday 23 April Today 38° 25° Fri 38° 26° Sat 39°
26° Sun 39° 24° Pune 37°C Google Weather Top stories The Hindu More
Israel-Iran LIVE: Iran attacks 3 ships in Strait of Hormuz 1 hour ago
NDTV More India-Bound Ship Among 2 Seized By Iran's Revolutionary
Guards 14 hours ago By Subham Tiwari BBC More Iran says it has seized
two ships in Strait of Hormuz after vessels attacked 10 hours ago By
Thomas Mackintosh AP News More Iran attacks 3 ships in the Strait of
Hormuz as Trump indefinitely ex

In [10]:
all_tools = tools + [tool_push]

In [11]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools = llm.bind_tools(all_tools)


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


In [12]:

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot -.-> __end__;
	chatbot -.-> tools;
	tools --> chatbot;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [13]:
import asyncio

config = {"configurable": {"thread_id": "10"}}

def chat(user_input: str, history):
    loop = asyncio.get_event_loop()
    result = loop.run_until_complete(
        graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    )
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\AI_AGENTIC_COURSE_main\agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py", line 3071, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_AGENTIC_COURSE_main\agents\.venv\Lib\site-packages\google\genai\models.py", line 6261, in generate_content
    return self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_AGENTIC_COURSE_main\agents\.venv\Lib\site-packages\google\genai\models.py", line 4730, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_AGENTIC_COURSE_main\agents\.venv\Lib\site-packages\google\genai\_api_client.py", line 1537, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_AGENTIC_COURSE